## Getting Line Upper Bounds

Here, we estimate 3 sigma uper bounds for emission and absorption lines by comparing flux uncertainties before and after fitting using bootstrapping with autocorrelation. We generate a distribution of correction factors (ratios of errors using curve_fit and using full bootstrapping) and then set an appropriate value to estimate the "true" flux errors of lines that only ever get curve_fit fits.

In [ ]:
from astropy.io import fits
import astropy.table as aptb
import matplotlib.pyplot as plt
import numpy as np

# Load the megatab
SPEC_SOURCE = "R21"  #  Where are the spectra from? 'R21' for Richard+21, 'APER' for apertures
SPEC_TYPE   = "weight_skysub"

tabfiles = {
    'before': f"megatables/fit_catalog_qc_cpts_stack_zeldamcmc_{SPEC_TYPE}.fits",
    'after': f"megatables/fit_catalog_qc_cpts_stack_zeldamcmc_refit_sysv_absv_{SPEC_TYPE}.fits"
}

tabhduls = {key: fits.open(path) for key, path in tabfiles.items()}

tabs = {key: aptb.Table(hdul[1].data) for key, hdul in tabhduls.items()}

tabs['before'] = tabs['before'][(tabs['before']['SNRR'] > 5) | (tabs['before']['SNRB'] > 5)]

# Ensure tables are sorted in the exact same way
for key in tabs.keys():
    tabs[key].sort('iden')

# Verify that the tables are identical in terms of rows
assert np.array_equal(tabs['before']['iden'], tabs['after']['iden']), "The tables have different rows or are in a different order!"
assert np.array_equal(tabs['before']['CLUSTER'], tabs['after']['CLUSTER']), "The tables have different rows or are in a different order!"

In [ ]:
# For each line with SNR > 3 in the before tab, calculate the ratio of the FLUX_ERR_{line} in the after to the before table

lines = [
    'CIII1907', 'CIII1909',
    'HeII1640',
    'CIV1548', 'CIV1551',
    'OIII1660', 'OIII1666'
]

error_ratios = [] # initialise list to hold error ratios

for line in lines:
    # Significant detections in the before table
    sig_before = np.abs(tabs['before'][f'SNR_{line}']) > 3
    # Get the flux errors for the line in both tables
    err_before = tabs['before'][f'FLUX_ERR_{line}'][sig_before]
    err_after = tabs['after'][f'FLUX_ERR_{line}'][sig_before]
    
    # Calculate the ratio of the errors
    ratio = err_after / err_before
    
    error_ratios.extend(ratio)



In [ ]:
# Find the mean, median and 90th percentile of the error ratios
mean_ratio = np.mean(error_ratios)
median_ratio = np.median(error_ratios)
percentile_90 = np.percentile(error_ratios, 90)


# Now plot histogram of error ratios
plt.figure(figsize=(8,6))
plt.hist(np.log10(error_ratios), bins=20, color='skyblue', edgecolor='black')
plt.xlabel('Log10(Error Ratio) (After / Before)')
plt.ylabel('Number of Lines')
plt.title('Distribution of Error Ratios for Significant Lines')
plt.axvline(0, color='red', linestyle='dashed', linewidth=1)
plt.axvline(np.log10(mean_ratio), color='purple', linestyle='dashed', linewidth=1)
plt.axvline(np.log10(median_ratio), color='blue', linestyle='dashed', linewidth=1)
plt.axvline(np.log10(percentile_90), color='green', linestyle='dashed', linewidth=1)
plt.text(0.05, plt.ylim()[1]*0.9, 'Ratio = 1', color='red')
plt.text(np.log10(mean_ratio), plt.ylim()[1]*0.85, f'Mean = {mean_ratio:.2f}', color='purple')
plt.text(np.log10(median_ratio), plt.ylim()[1]*0.8, f'Median = {median_ratio:.2f}', color='blue')
plt.text(np.log10(percentile_90), plt.ylim()[1]*0.7, f'90th Percentile = {percentile_90:.2f}', color='green')
# plt.xscale('log')
plt.show()

In [ ]:
# Now add upper bounds to the final table for lines that were not significant in the before table, using
# the before error vlaues multiplied by 3 times a chosen ratio (e.g. the mean or median ratio from above)

chosen_ratio = 2 # 2 is very close to the mean, and represents a reasonable conservative correction factor

# Get ALL lines from the table column names
line_names = set()
for col in tabs['before'].colnames:
    if col.startswith('SNR_'):
        line_names.add(col[4:]) # Remove 'SNR_' prefix to get line name

tabs['final'] = tabs['after'].copy() # Create a copy of the after table to modify

for line in line_names:
    # Add a FLUX_UB_{line} column to the final table
    tabs['final'][f'FLUX_UB_{line}'] = np.nan # Initialize with NaN values
    # Significant detections in the before table
    sig_before = np.abs(tabs['before'][f'SNR_{line}']) > 3
    notnan_before = ~np.isnan(tabs['before'][f'SNR_{line}'])
    mask = ~sig_before & notnan_before
    # For lines that were NOT significant in the before table, multiply the error in the after table by the chosen ratio
    tabs['final'][f'FLUX_UB_{line}'][mask] = 3 * chosen_ratio * tabs['before'][f'FLUX_ERR_{line}'][mask]
    print(f"Added upper bounds for {line} in the final table for {np.sum(mask)} lines that were not significant in the before table.")

# Check that updates were applied correctly for one line (e.g. CIII1907)
line = 'CIII1907'
sig_before = np.abs(tabs['before'][f'SNR_{line}']) > 3
notnan_before = ~np.isnan(tabs['before'][f'SNR_{line}'])
mask = ~sig_before & notnan_before
updated_ubs = tabs['final'][f'FLUX_UB_{line}'][mask]
original_errors = tabs['before'][f'FLUX_ERR_{line}'][mask]
print(f"Original errors for non-significant lines in the before table for {line}: {original_errors}")
print(f"Upper bounds for non-significant lines in the final table for {line}: {updated_ubs}")
ratio_check = updated_ubs / original_errors
print(f"Ratio of updated upper bounds to original errors for non-significant lines in {line}: {ratio_check}")

In [ ]:
# Save the final table to a new FITS file
writename = f"megatables/fit_catalog_qc_cpts_stack_zeldamcmc_refit_sysv_absv_ubs_{SPEC_TYPE}.fits"

tabs['final'].write(writename, overwrite=True)